In [20]:
import pandas as pd
import numpy as np

In [4]:
df2=pd.read_csv("../data/synthetic.csv")
df2

,patient_id,note,question,answer,task
0,0,Discharge Summary:\n\nPatient: 60-year-old mal...,Can you provide a simplified paraphrase of the...,The healthcare team used a gradual approach to...,Paraphrasing
1,1,Discharge Summary:\n\nAdmission Date: [Insert ...,Which coreferences were resolved in the hospit...,The hospital course section resolved the coref...,Coreference Resolution
2,2,Hospital Course Summary:\n\nAdmission Date: [I...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization
3,3,Discharge Summary:\n\nPatient: 69-year-old mal...,What roles did physical therapists have in the...,Physical therapists were responsible for ensur...,Relation Extraction
4,4,Discharge Summary:\n\nPatient Information:\n- ...,What manual airway clearance techniques were u...,The discharge summary stated that 1-2 physical...,Relation Extraction
...,...,...,...,...,...
158109,167029,Patient Background:\n- A 30-year-old male with...,What is the relationship between the patient's...,The prescribed Methimazole and atenolol treatm...,Relation Extraction
158110,167030,Discharge Summary:\n\nPatient Name: Not Provid...,What instances of coreference can be identifie...,The instances of coreference related to the pa...,Coreference Resolution
158111,167031,Discharge Summary\n\nPatient Name: [Redacted]\...,What diagnostic test revealed the mediastinal ...,The mediastinal mass was revealed by computed ...,Temporal Information Extraction
158112,167032,Patient Information: \nThe patient is a woman ...,What named entities related to the patient's s...,"Using Named Entity Recognition, the named enti...",Named Entity Recognition


In [13]:
import sys
!"{sys.executable}" -m pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
df_QA=pd.read_csv("../data/synthetic.csv")
df_QA.head()

,patient_id,note,question,answer,task
0,0,Discharge Summary:\n\nPatient: 60-year-old mal...,Can you provide a simplified paraphrase of the...,The healthcare team used a gradual approach to...,Paraphrasing
1,1,Discharge Summary:\n\nAdmission Date: [Insert ...,Which coreferences were resolved in the hospit...,The hospital course section resolved the coref...,Coreference Resolution
2,2,Hospital Course Summary:\n\nAdmission Date: [I...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization
3,3,Discharge Summary:\n\nPatient: 69-year-old mal...,What roles did physical therapists have in the...,Physical therapists were responsible for ensur...,Relation Extraction
4,4,Discharge Summary:\n\nPatient Information:\n- ...,What manual airway clearance techniques were u...,The discharge summary stated that 1-2 physical...,Relation Extraction


In [30]:
print("null note",df_QA["note"].isna().unique())
print("null question",df_QA["question"].isna().unique())
print("null answer",df_QA["answer"].isna().unique())
print("null task",df_QA["task"].isna().unique())

null note [False]
null question [False]
null answer [False]
null task [False]


In [31]:
print(len(df_QA))

158114


In [32]:
print(len(df_QA[df_QA["task"]=="Summarization"]))

19756


In [33]:
print(len(df_QA["task"].unique()))

8


In [45]:
ssss=df_QA[df_QA["task"]=="Summarization"]["question"][:10]

In [46]:
for s in ssss:
    print(s)

What were the key improvements in the patient's medical condition during the hospital course, and how was physical therapy utilized to achieve these results?
How did the patient's treatment for dysphagia progress during their hospital stay, as outlined in the discharge summary?
Can you provide a summary of the treatment, hospital course, and post-discharge plan for a 45-year-old female patient with a history of restrictive AN, binge-purge behavior, old traumatic brain injury, and cholecystitis, according to the provided discharge summary?
Based on the given discharge summary, can you summarize the patient's treatment plan for managing their breast cancer, including recommended medications and the rationale for switching from goserelin to triptorelin?
What are the key findings and diagnosis of the patient with cutaneous T-cell lymphoma/mycosis fungoides, type II diabetes mellitus, atrial flutter, sick sinus syndrome, and metastatic Merkel cell carcinoma in the given discharge summary?
W

In [43]:
import pandas as pd
import re


# 2. Fonction de nettoyage pour les textes (notes et réponses)
def clean_medical_text(text):
    if not isinstance(text, str):
        return ""
    # Remplacer les retours à la ligne (\n) et tabulations par un espace
    text = re.sub(r'[\n\t\r]+', ' ', text)
    # Remplacer les espaces multiples par un seul espace
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# Application du nettoyage
print("Nettoyage de la colonne 'note'...")
df_QA['note'] = df_QA['note'].apply(clean_medical_text)

print("Nettoyage de la colonne 'question'...")
df_QA['question'] = df_QA['question'].astype(str).str.strip()

print("Nettoyage de la colonne 'answer'...")
df_QA['answer'] = df_QA['answer'].apply(clean_medical_text)

# 3. Filtrage critique : Supprimer les lignes où 'answer' est trop courte ou tronquée
# Par exemple, on peut supposer qu'une réponse valide fait au moins 20 caractères
# et se termine par un point de ponctuation final (., !, ?)
def is_valid_answer(text):
    if len(text) < 20:
        return False
    # Vérifie si le dernier caractère est une ponctuation de fin de phrase
    if text[-1] not in ['.', '!', '?']:
        return False
    return True

taille_avant = len(df_QA)
df_QA = df_QA[df_QA['answer'].apply(is_valid_answer)].copy()
taille_apres = len(df_QA)

print(f"Nettoyage terminé ! {taille_avant - taille_apres} lignes suspectes/tronquées ont été retirées.")

Nettoyage de la colonne 'note'...
Nettoyage de la colonne 'question'...
Nettoyage de la colonne 'answer'...
Nettoyage terminé ! 5808 lignes suspectes/tronquées ont été retirées.


In [44]:
print(len(df_QA))

152306


In [47]:
df_QA=df_QA[df_QA["task"]=="Summarization"]

In [48]:
print(len(df_QA))

19746


In [49]:
df_test=df_QA[:10]

In [51]:
df_test.head(11)

,patient_id,note,question,answer,task
2,2,Hospital Course Summary: Admission Date: [Inse...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization
5,5,Discharge Summary: Patient: 52-year-old male h...,How did the patient's treatment for dysphagia ...,"During the patient's hospital stay, treatment ...",Summarization
10,11,Discharge Summary: Patient Name: [REDACTED] Me...,"Can you provide a summary of the treatment, ho...",The 45-year-old female patient with a history ...,Summarization
12,13,DISCHARGE SUMMARY: Patient Name: X Medical Rec...,"Based on the given discharge summary, can you ...",The patient with a multifocal invasive mammary...,Summarization
18,20,Hospital Course: The patient is a 78-year-old ...,What are the key findings and diagnosis of the...,The key findings of the patient include abnorm...,Summarization
33,36,"Hospital Course: The patient, a 35-year-old ma...",What radiological and clinical findings led to...,Suspected SAPHO syndrome was raised based on r...,Summarization
41,44,Discharge Summary Patient Name: [redacted] Dat...,What were the key clinical findings and diagno...,The 68-year-old female patient was diagnosed w...,Summarization
44,47,Discharge Summary: Patient Name: [Redacted] Me...,What is a brief summary of the patient's hospi...,The patient was admitted with a severe stroke ...,Summarization
45,48,Hospital Course Summary: Patient has been admi...,What were the patient's mobility and functiona...,The patient had significant lower extremity we...,Summarization
58,63,Hospital Course: This is a summary of the hosp...,What was the hospital course and diagnosis of ...,The 5-year-old boy's hospital course included ...,Summarization


In [52]:
df_test.to_csv("../data/test_dataset.csv")

In [53]:
df_QA.to_csv("../data/QA_dataset.csv")

In [54]:
df_last=df_QA[:1500]

In [55]:
print(len(df_last))

1500


In [56]:
df_last.to_csv("../data/QA_Dataset.csv")